# 第105章 ROC、PR曲线与决策阈值

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 20 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 混淆矩阵与分类指标  →  **本章任务：** ROC、PR曲线与决策阈值  →  **下一步：** 类别不平衡与Top-K Lift
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

光看“模型准不准”其实不够，很多真实难题更关心“把谁挑出来、会不会误伤”。


## 本章目标

学完本章，你将能够：

- **理解**：理解「ROC、PR曲线与决策阈值」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「ROC、PR曲线与决策阈值」的关键输出指标。
- **迁移**：能把「ROC、PR曲线与决策阈值」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：光看“模型准不准”其实不够，很多真实难题更关心“把谁挑出来、会不会误伤”。比如风控要把有风险的人识别出来、医疗要做筛查，漏掉一个重要案例和误报一个正常案例的代价并不相同。ROC、PR 曲线和决策阈值，就是帮你把这种“取舍”画成看得懂的表和曲线，在召回率与精确率之间找到适合业务成本的判断刻度。下面先记住四个核心量，零基础也能顺着一步步做出来。

- TPR=TP/(TP+FN)
- FPR=FP/(FP+TN)
- PR-AUC 更关注正类稀少任务
- 阈值选择不能依赖最终测试集（打个比方：阈值像调“警铃灵敏度”，得用没当过裁判的验证数据来定；拿最终测试集来挑，等于先偷看了答案再报成绩。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `model.predict_proba()`、`.fit()` | 先明确样本、特征、目标和验证方式，再训练模型。 | AUC 高就忽略阈值表现 |
| 模型、公式与诊断 | `np.arange()`、`rows.append()`、`p.mean()`、`pd.DataFrame()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 在测试集上挑阈值 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-105 -->
### 数学推导｜ROC 与 PR 曲线的坐标

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜移动阈值。** 每个 $t$ 都产生一组 $TP(t),FP(t),FN(t),TN(t)$。

**第 2 步｜计算曲线坐标。** ROC 使用 $(FPR(t),TPR(t))$，PR 使用 $(Recall(t),Precision(t))$。

**第 3 步｜理解面积。** ROC-AUC 可解释为随机抽取一个正例和负例时，正例得分更高的概率：

$$
AUC=P(score^+>score^-)
$$

但面积平均了所有阈值，最终行动点仍需结合基准率、预算和错误成本选择。

**把上面的关系收束为本章计算式：**

$$
TPR=\frac{TP}{TP+FN},\qquad FPR=\frac{FP}{FP+TN},\qquad Precision=\frac{TP}{TP+FP}
$$

**符号解释：** ROC 绘制 TPR–FPR，PR 绘制 Precision–Recall。

**代码对应：** 类别稀少时优先结合 PR-AUC、基准正类率和实际行动量。

**使用边界：** AUC 汇总所有阈值，不直接给出部署阈值或业务收益。


In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=94
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    X_train, y_train
)
prob = model.predict_proba(X_test)[:, 1]


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：示例 1 在乳腺数据上训练了逻辑回归，拿到每个测试样本的预测概率 `prob`。不过在挑阈值之前，先要看清数据本身偏不偏——这决定了该用 ROC 还是更关注 PR。请把下面 `n_pos`、`n_neg` 两个空补上，统计目标变量 `y` 中正类(1)与负类(0)各有多少，填好后运行自检。提示：`np.sum(y)` 就是正类个数，`(y==0)` 的和就是负类个数，两者相加应等于样本总数 `len(y)`。


In [ ]:
try:
    # 请在下方填写代码
    # 目标：统计目标变量 y 中正类(1)与负类(0)的样本数量，并观察类别是否均衡。
    # 提示：np.sum(y) 统计正类个数；(y==0) 的和统计负类个数。

    # --- 你的代码 ---
    n_pos = None  # 请填：统计正类(1)个数的代码
    n_neg = None  # 请填：统计负类(0)个数的代码
    # --- 你的代码结束 ---

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

print(
    "ROC-AUC/PR-AUC:",
    round(roc_auc_score(y_test, prob), 3),
    round(average_precision_score(y_test, prob), 3),
)
rows = []
for t in np.arange(0.1, 1, 0.1):
    p = prob >= t
    rows.append(
        [t, precision_score(y_test, p), recall_score(y_test, p), p.mean()]
    )
thresholds = pd.DataFrame(
    rows, columns=["threshold", "precision", "recall", "positive_rate"]
)
display(thresholds.round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
_demo_model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(_demo_model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, _demo_model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = _demo_model.predict(X_changed)
print("原始前2个预测：", np.round(_demo_model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(changed_prediction[:2] - _demo_model.predict(X[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- AUC 高就忽略阈值表现
- 在测试集上挑阈值
- 正类比例变化后仍沿用旧 PR 基线
- 没有考虑复核容量


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 105.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 105.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 105.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用 ROC、PR 曲线和阈值表评估概率排序，并根据成本选择决策阈值。


### 你已经掌握

- 计算 ROC-AUC 与 PR-AUC
- 理解 TPR、FPR 和 Precision
- 生成阈值性能表
- 按错误成本选择阈值


### 需要注意

- AUC 高就忽略阈值表现
- 在测试集上挑阈值
- 正类比例变化后仍沿用旧 PR 基线
- 没有考虑复核容量


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 完整答案
n_pos = int(np.sum(y))  # 正类(1)的数量
n_neg = int(len(y) - np.sum(y))  # 负类(0)的数量


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
eligible = thresholds.query("recall>=0.95")
selected_threshold = float(
    eligible.loc[eligible.precision.idxmax(), "threshold"]
)
print("满足召回约束的高精确率阈值:", selected_threshold)
